# Thailand Oil → CPI Pass-Through Analysis

This notebook estimates how **oil price shocks** transmit into **Thailand headline CPI YoY inflation** using a **Local Projections** approach (Jordà, 2005).

## Outputs
- Impulse response function (1–8 quarters)
- Scenario table for +10%, +20%, +30% oil shocks
- CSV exports for further analysis

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

# File paths (adjust if running from different directory)
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")

BRENT_FILE = DATA_DIR / "brent_quarterly.xlsx"
CPI_FILE = DATA_DIR / "thai_cpi_quarterly.xlsx"

## 2. Load Data

In [ ]:
def read_any(path):
    """Read CSV or Excel file."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.resolve()}")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    raise ValueError("Use .csv or .xlsx")

brent_raw = read_any(BRENT_FILE)
cpi_raw = read_any(CPI_FILE)

print("Brent columns:", brent_raw.columns.tolist())
print("CPI columns:", cpi_raw.columns.tolist())
print()
display(brent_raw.head())
display(cpi_raw.head())

## 3. Process Quarterly Data

In [ ]:
# Column name mappings
BRENT_DATE_COL = "Date"
BRENT_VALUE_COL = "Brent_Oil_USD_per_Barrel"
CPI_DATE_COL = "Date"
CPI_VALUE_COL = "CPI_Index"

def normalize_quarter_label(x) -> str:
    """Normalize quarter format to YYYY-Q#."""
    s = str(x).strip().upper().replace(" ", "")
    # Convert 2016Q1 -> 2016-Q1
    if len(s) == 6 and s[:4].isdigit() and s[4] == "Q" and s[5] in "1234":
        s = f"{s[:4]}-Q{s[5]}"
    # Validate format
    if not (len(s) == 7 and s[:4].isdigit() and s[4:6] == "-Q" and s[6] in "1234"):
        raise ValueError(f"Bad quarter format: {x} (expected like 2016-Q1 or 2016Q1)")
    return s

def quarter_sort_key(q: str):
    """Sort key for quarter strings."""
    return (int(q[:4]), int(q[-1]))

# Process Brent data
brent_q = brent_raw[[BRENT_DATE_COL, BRENT_VALUE_COL]].copy()
brent_q["Date"] = brent_q[BRENT_DATE_COL].apply(normalize_quarter_label)
brent_q["brent"] = pd.to_numeric(brent_q[BRENT_VALUE_COL], errors="coerce")
brent_q = brent_q[["Date", "brent"]].dropna()
brent_q = brent_q.sort_values("Date", key=lambda s: s.map(quarter_sort_key)).reset_index(drop=True)

# Process CPI data
cpi_q = cpi_raw[[CPI_DATE_COL, CPI_VALUE_COL]].copy()
cpi_q["Date"] = cpi_q[CPI_DATE_COL].apply(normalize_quarter_label)
cpi_q["cpi"] = pd.to_numeric(cpi_q[CPI_VALUE_COL], errors="coerce")
cpi_q = cpi_q[["Date", "cpi"]].dropna()
cpi_q = cpi_q.sort_values("Date", key=lambda s: s.map(quarter_sort_key)).reset_index(drop=True)

# Calculate YoY inflation (4-quarter change)
cpi_q["YoY_Inflation_%"] = cpi_q["cpi"].pct_change(4) * 100

print(f"Brent: {brent_q['Date'].min()} to {brent_q['Date'].max()} ({len(brent_q)} quarters)")
print(f"CPI:   {cpi_q['Date'].min()} to {cpi_q['Date'].max()} ({len(cpi_q)} quarters)")

## 4. Build Modeling Dataset

In [ ]:
# Merge datasets
df = pd.merge(brent_q, cpi_q, on="Date", how="inner").copy()

# Define oil shock as log change (× 100 = % change)
df["oil_shock_pct"] = 100 * (np.log(df["brent"]) - np.log(df["brent"].shift(1)))

# Drop rows with missing values
df = df.dropna(subset=["YoY_Inflation_%", "oil_shock_pct"]).reset_index(drop=True)

print(f"Modeling dataset: {len(df)} observations")
print(f"Period: {df['Date'].iloc[0]} to {df['Date'].iloc[-1]}")
df.head(10)

## 5. Local Projections (Impulse Response)

For each horizon $h = 1, ..., 8$:

$$\text{CPI YoY}_{t+h} = \alpha_h + \beta_h \cdot \text{oil\_shock}_t + \sum_{k=1}^{4} \gamma_k \cdot \text{CPI YoY}_{t-k} + \sum_{k=1}^{4} \delta_k \cdot \text{oil\_shock}_{t-k} + \varepsilon_{t+h}$$

- HAC (Newey–West) standard errors with maxlags = $h + 1$
- $\beta_h$ = CPI YoY response (pp) per 1% oil shock at horizon $h$

In [ ]:
HORIZONS = list(range(1, 9))  # 1 to 8 quarters
LAGS_Y = 4  # Lags of CPI YoY
LAGS_X = 4  # Lags of oil shock

# Create lag variables
for k in range(1, LAGS_Y + 1):
    df[f"y_lag{k}"] = df["YoY_Inflation_%"].shift(k)
for k in range(1, LAGS_X + 1):
    df[f"x_lag{k}"] = df["oil_shock_pct"].shift(k)

# Create lead variables (future outcomes)
for h in HORIZONS:
    df[f"y_lead{h}"] = df["YoY_Inflation_%"].shift(-h)

# Run local projections
results = []

for h in HORIZONS:
    X_cols = (["oil_shock_pct"]
              + [f"y_lag{k}" for k in range(1, LAGS_Y + 1)]
              + [f"x_lag{k}" for k in range(1, LAGS_X + 1)])
    
    tmp = df[["Date", f"y_lead{h}"] + X_cols].dropna().copy()
    
    y = tmp[f"y_lead{h}"]
    X = sm.add_constant(tmp[X_cols])
    
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": h + 1})
    
    beta = model.params["oil_shock_pct"]
    se = model.bse["oil_shock_pct"]
    results.append({
        "h": h,
        "beta": float(beta),
        "se": float(se),
        "n": int(model.nobs)
    })

# Build IRF table
irf = pd.DataFrame(results)
irf["lower"] = irf["beta"] - 1.96 * irf["se"]
irf["upper"] = irf["beta"] + 1.96 * irf["se"]
irf["significant"] = (irf["lower"] > 0) | (irf["upper"] < 0)

irf

## 6. Plot Impulse Response Function

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(irf["h"], irf["beta"], marker="o", linewidth=2, markersize=8, label="Point estimate")
ax.fill_between(irf["h"], irf["lower"], irf["upper"], alpha=0.2, label="95% CI")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

ax.set_xticks(irf["h"])
ax.set_xlabel("Horizon (quarters)", fontsize=12)
ax.set_ylabel("CPI YoY response (pp) per 1% oil shock", fontsize=12)
ax.set_title("Thailand CPI Pass-Through from Oil Price Shocks\n(Local Projections with HAC Standard Errors)", fontsize=14)
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "irf_plot.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Scenario Analysis

In [ ]:
SHOCKS = [10, 20, 30]  # Percentage oil price shocks

rows = []
for shock in SHOCKS:
    for _, r in irf.iterrows():
        rows.append({
            "Shock_%": shock,
            "Horizon_q": int(r["h"]),
            "CPI_Impact_pp": round(r["beta"] * shock, 3),
            "Lower_pp": round(r["lower"] * shock, 3),
            "Upper_pp": round(r["upper"] * shock, 3),
        })

scenario_table = pd.DataFrame(rows)

# Display 20% shock scenario
print("Scenario: 20% Oil Price Shock")
print("="*50)
scenario_table[scenario_table["Shock_%"] == 20].reset_index(drop=True)

## 8. Export Results

In [ ]:
# Ensure results directory exists
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Export IRF
irf_export = irf[["h", "beta", "se", "lower", "upper", "n"]]
irf_export.to_csv(RESULTS_DIR / "irf_oil_to_cpi_quarterly.csv", index=False)

# Export scenarios
scenario_table.to_csv(RESULTS_DIR / "scenario_impacts_oil_to_cpi_quarterly.csv", index=False)

print(f"Saved: {RESULTS_DIR / 'irf_oil_to_cpi_quarterly.csv'}")
print(f"Saved: {RESULTS_DIR / 'scenario_impacts_oil_to_cpi_quarterly.csv'}")

## 9. Methods Summary

| Component | Description |
|-----------|-------------|
| **Model** | Local Projections (Jordà, 2005) |
| **Shock** | Quarterly % change in Brent oil price (log change × 100) |
| **Outcome** | Thailand headline CPI YoY inflation |
| **Controls** | 4 lags of CPI YoY, 4 lags of oil shocks |
| **Inference** | HAC (Newey–West) standard errors, maxlags = h+1 |
| **Sample** | 2017-Q1 to 2025-Q3 (quarterly) |

> **Note:** This is a reduced-form historical pass-through estimate, not a structural forecast.